# Практична робота №12 — Імітаційне моделювання та оптимізація стохастичної системиВкажи номер варіанта, коли клітинка попросить. Система — багатоканальна СМО з очікуванням (M/M/c): заявки надходять найпростішим (пуасонівським) потоком з інтенсивністю λ, обслуговуються c паралельними каналами з експоненційним часом обслуговування (інтенсивність μ одного каналу). Мета — узгодити аналітичну модель (формули Ерланга) з імітаційною (метод Монте-Карло) і знайти оптимальну кількість каналів c* за критерієм сумарних витрат C(c) = Cs·c + Cw·Lq(c).

In [ ]:
import mathimport heapqimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as plt

In [ ]:
from variants import VARIANTS_default = globals().get("VARIANT", 1)_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()VARIANT = int(_raw) if _raw else _defaultassert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}" 

## 1. Постановка задачіЗ таблиці варіантів дістаємо λ (інтенсивність потоку заявок), μ (інтенсивність обслуговування одним каналом), Cs (вартість утримання каналу за годину) і Cw (втрати від очікування заявки за годину очікування), а також змістовний об'єкт моделювання. Заявка — клієнт/запит, що звертається до системи; канал — один обслуговуючий ресурс (оператор, пост, сервер); втрати від очікування — грошовий еквівалент того, що заявка простоює в черзі, а не обслуговується.Мінімально допустима кількість каналів визначається з умови стійкості ρ = λ/(c·μ) < 1, інакше черга необмежено зростає.

In [ ]:
LAM, MU, C_SERV, C_WAIT, OBJECT = VARIANTS[VARIANT]LAM = float(LAM); MU = float(MU); C_SERV = float(C_SERV); C_WAIT = float(C_WAIT)c_min = math.ceil(LAM / MU)print(f"Варіант {VARIANT}: об'єкт моделювання — {OBJECT}")print(f"λ = {LAM} заявок/год, μ = {MU} заявок/год (на канал)")print(f"Cs = {C_SERV} грн/год (канал), Cw = {C_WAIT} грн/год очікування (заявка)")print(f"Мінімально допустима кількість каналів: c_min = ceil(λ/μ) = ceil({LAM/MU:.3f}) = {c_min}")

## 2. Аналітична модель M/M/c (формули Ерланга)Для стаціонарного режиму (ρ < 1):p0 = [ Σ_{k=0}^{c−1} a^k/k! + a^c / (c!(1−ρ)) ]⁻¹, a = λ/μ, ρ = a/cLq = p0·a^c·ρ / (c!·(1−ρ)²),  Wq = Lq/λ,  W = Wq + 1/μ,  L = Lq + aРозглядаємо c від c_min до c_min + 6.

In [ ]:
def mmc(lam, mu, c):    """Аналітичні показники M/M/c за формулами Ерланга (None, якщо ρ >= 1)."""    rho = lam / (c * mu)    if rho >= 1:        return None    a = lam / mu    s = sum(a ** k / math.factorial(k) for k in range(c))    p0 = 1 / (s + a ** c / (math.factorial(c) * (1 - rho)))    Lq = p0 * a ** c * rho / (math.factorial(c) * (1 - rho) ** 2)    Wq = Lq / lam    return dict(rho=rho, p0=p0, Lq=Lq, Wq=Wq, L=Lq + a, W=Wq + 1 / mu)c_range = list(range(c_min, c_min + 7))analytic = {c: mmc(LAM, MU, c) for c in c_range}df_analytic = pd.DataFrame([    dict(c=c, rho=a["rho"], p0=a["p0"], Lq=a["Lq"], Wq_hr=a["Wq"], Wq_min=a["Wq"] * 60, L=a["L"], W_hr=a["W"])    for c, a in analytic.items()])df_analytic

## 3. Перевірка формули ЛіттлаФормула Літтла: L = λ·W (середня кількість заявок у системі = інтенсивність потоку × середній час перебування). Перевіряємо її для кожного c з аналітичної таблиці — розбіжність має бути на рівні похибки округлення.

In [ ]:
rows = []for c, a in analytic.items():    L_calc = LAM * a["W"]    rows.append(dict(c=c, L_table=a["L"], L_from_Little=L_calc, diff=abs(a["L"] - L_calc)))df_little = pd.DataFrame(rows)df_little

## 4. Імітаційна модель (дискретно-подійна схема, метод Монте-Карло)Генеруємо моменти надходження заявок як накопичену суму експоненційних інтервалів (пуасонівський потік) і тривалості обслуговування — експоненційні з інтенсивністю μ. Кожна заявка займає канал, який звільняється найшвидше (мін-купа `free` з c нулів). Початкова частка прогону (warm-up) відкидається, щоб не враховувати перехідний режим. Довірчий інтервал для Wq: x̄ ± 1.96·s/√N (нормальне наближення для великого N).

In [ ]:
def simulate(lam, mu, c, n_jobs=50_000, seed=0, warm_frac=0.1):    """Дискретно-подійна імітація M/M/c. Повертає оцінки Wq, Lq, p_wait і 95% ДІ для Wq."""    rng = np.random.default_rng(seed)    arr = np.cumsum(rng.exponential(1 / lam, n_jobs))    srv = rng.exponential(1 / mu, n_jobs)    free = [0.0] * c    heapq.heapify(free)    wait = np.empty(n_jobs)    for i in range(n_jobs):        t = heapq.heappop(free)        start = max(arr[i], t)        wait[i] = start - arr[i]        heapq.heappush(free, start + srv[i])    warm = int(n_jobs * warm_frac)    w = wait[warm:]    ci = 1.96 * w.std(ddof=1) / math.sqrt(len(w))    return dict(Wq=w.mean(), Lq=w.mean() * lam, p_wait=float((w > 1e-12).mean()), ci=ci, n_used=len(w))# Перевірка на малому прикладіtest = simulate(LAM, MU, c_min + 2, n_jobs=20_000, seed=1)print(test)

## 5. Валідація: порівняння аналітичної та імітаційної моделейПослідовні часи очікування в черзі в межах одного прогону сильно автокорельовані (особливо при ρ близькому до 1), тому наївний довірчий інтервал 1.96·s/√N, обчислений по одному прогону так, ніби спостереження незалежні, занижує справжню невизначеність (легко перевірити: середнє Wq по кількох прогонах з різними seed розкидається значно сильніше, ніж «обіцяє» такий ДІ). Коректний спосіб — незалежні реплікації: запускаємо по кожному c кілька прогонів з різними seed, усереднюємо Wq і будуємо ДІ по розкиду цих середніх.

In [ ]:
def replicated_wq(lam, mu, c, n_reps=30, n_jobs=30_000, seed0=9000):    means = np.array([simulate(lam, mu, c, n_jobs=n_jobs, seed=seed0 + c * 1000 + i)["Wq"] for i in range(n_reps)])    return means.mean(), 1.96 * means.std(ddof=1) / math.sqrt(n_reps)rows = []for c in c_range:    a = analytic[c]    wq_sim, ci = replicated_wq(LAM, MU, c)    diff = abs(a["Wq"] - wq_sim)    rows.append(dict(c=c, rho=a["rho"], Wq_theory_hr=a["Wq"], Wq_sim_hr=wq_sim, ci95=ci,                      diff=diff, within_ci=diff <= ci))df_validation = pd.DataFrame(rows)df_validation

## 6. Оптимізація: функція сумарних витрат C(c)C(c) = Cs·c + Cw·Lq(c) → min, c ≥ c_min. Перший доданок зростає лінійно (більше каналів — вища вартість утримання), другий спадає гіперболічно (більше каналів — менша черга), тому сума опукла і має єдиний мінімум. Оскільки c ціле, достатньо прямого перебору у вузькому діапазоні.

In [ ]:
cost_rows = []for c in c_range:    a = analytic[c]    cost = C_SERV * c + C_WAIT * a["Lq"]    cost_rows.append(dict(c=c, cost_servers=C_SERV * c, cost_queue=C_WAIT * a["Lq"], cost_total=cost))df_cost = pd.DataFrame(cost_rows)best_row = df_cost.loc[df_cost["cost_total"].idxmin()]c_star = int(best_row["c"])print(df_cost)print(f"\nОптимальна кількість каналів: c* = {c_star}, сумарні витрати = {best_row['cost_total']:.1f} грн/год")fig, ax = plt.subplots(figsize=(7, 5))ax.plot(df_cost["c"], df_cost["cost_total"], "o-", label="C(c), сумарні витрати")ax.plot(df_cost["c"], df_cost["cost_servers"], "s--", alpha=.6, label="Cs·c, вартість каналів")ax.plot(df_cost["c"], df_cost["cost_queue"], "^--", alpha=.6, label="Cw·Lq, втрати черги")ax.axvline(c_star, color="red", ls=":", label=f"c* = {c_star}")ax.set_xlabel("c, кількість каналів"); ax.set_ylabel("витрати, грн/год")ax.set_title(f"Варіант {VARIANT}: функція сумарних витрат")ax.legend(); fig.tight_layout()fig.savefig("cost_function.png", dpi=150)fig

## 7. Аналіз чутливості до λ і до співвідношення Cw/CsЗмінюємо λ на ±15% і ±30% від базового значення, а також подвоюємо співвідношення Cw/Cs (через подвоєння Cw), і дивимось, як змінюється оптимальна кількість каналів c* і завантаження ρ при оптимумі.

In [ ]:
def best_c(lam, mu, cs, cw, span=20):    cmin = math.ceil(lam / mu)    best = None    for c in range(cmin, cmin + span):        a = mmc(lam, mu, c)        if a is None:            continue        cost = cs * c + cw * a["Lq"]        if best is None or cost < best[1]:            best = (c, cost, a)    return bestsens_rows = []for label, lam_, cw_ in [    ("λ -30%", LAM * 0.70, C_WAIT), ("λ -15%", LAM * 0.85, C_WAIT), ("базовий", LAM, C_WAIT),    ("λ +15%", LAM * 1.15, C_WAIT), ("λ +30%", LAM * 1.30, C_WAIT),    ("Cw/Cs ×2", LAM, C_WAIT * 2),]:    c, cost, a = best_c(lam_, MU, C_SERV, cw_)    sens_rows.append(dict(scenario=label, lam=lam_, Cw=cw_, c_star=c, rho_star=a["rho"], cost=cost))df_sens = pd.DataFrame(sens_rows)df_sens

## 8. Чутливість до кількості реплікацій (точність оцінки Монте-Карло)Запускаємо не одну, а R незалежних реплікацій (різні seed) при c = c*, кожна довжиною n_jobs заявок, і будуємо довірчий інтервал методом батч-середніх (по середніх значеннях Wq кожної реплікації). Це показує, як звужується довірчий інтервал зі збільшенням числа реплікацій R — головна ідея методу Монте-Карло: більше статистики → точніша оцінка.

In [ ]:
def replications_ci(lam, mu, c, n_reps, n_jobs=5_000, seed0=1000):    means = np.array([simulate(lam, mu, c, n_jobs=n_jobs, seed=seed0 + i)["Wq"] for i in range(n_reps)])    m = means.mean()    ci = 1.96 * means.std(ddof=1) / math.sqrt(n_reps) if n_reps > 1 else float("nan")    return m, ci, meansrep_rows = []for R in (5, 10, 30, 100):    m, ci, _ = replications_ci(LAM, MU, c_star, R)    rep_rows.append(dict(R=R, Wq_mean_hr=m, ci95=ci, ci_width=2 * ci))df_reps = pd.DataFrame(rep_rows)df_reps

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))ax.plot(df_reps["R"], df_reps["ci_width"], "o-")ax.set_xscale("log")ax.set_xlabel("кількість реплікацій R (лог. шкала)")ax.set_ylabel("ширина 95% довірчого інтервалу, год")ax.set_title(f"Варіант {VARIANT}: звуження ДІ зі зростанням R (c = {c_star})")fig.tight_layout()fig.savefig("ci_narrowing.png", dpi=150)fig

## 9. Нелінійність черги: Lq(ρ)Фіксуємо кількість каналів c (беремо c_min) і змінюємо λ так, щоб ρ пробігало значення від 0,5 до 0,98, обчислюючи Lq за формулами Ерланга. Графік ілюструє, чому довжина черги зростає як 1/(1−ρ): при наближенні ρ до 1 крива йде у вертикальну асимптоту.

In [ ]:
rho_grid = np.linspace(0.5, 0.98, 25)lq_rows = []for rho in rho_grid:    lam_eq = rho * c_min * MU    a = mmc(lam_eq, MU, c_min)    lq_rows.append(dict(rho=rho, lam=lam_eq, Lq=a["Lq"], Wq_min=a["Wq"] * 60))df_nl = pd.DataFrame(lq_rows)fig, ax = plt.subplots(figsize=(6.5, 5))ax.plot(df_nl["rho"], df_nl["Lq"], "o-", ms=4)ax.set_xlabel("ρ, коефіцієнт завантаження"); ax.set_ylabel("Lq, середня довжина черги")ax.set_title(f"Варіант {VARIANT}: нелінійність Lq(ρ) при c = {c_min}")fig.tight_layout()fig.savefig("nonlinearity.png", dpi=150)fig

## 10. Додатково: інший розподіл часу обслуговування (M/G/c)Додаткове завдання: замінюємо експоненційний розподіл часу обслуговування на рівномірний і детермінований з тим самим середнім 1/μ — формул Ерланга для цього випадку не існує, тому висновок робимо лише з імітації.

In [ ]:
def simulate_general(lam, mu, c, dist, n_jobs=50_000, seed=0, warm_frac=0.1):    rng = np.random.default_rng(seed)    arr = np.cumsum(rng.exponential(1 / lam, n_jobs))    if dist == "exp":        srv = rng.exponential(1 / mu, n_jobs)    elif dist == "uniform":        srv = rng.uniform(0, 2 / mu, n_jobs)    elif dist == "deterministic":        srv = np.full(n_jobs, 1 / mu)    else:        raise ValueError(dist)    free = [0.0] * c    heapq.heapify(free)    wait = np.empty(n_jobs)    for i in range(n_jobs):        t = heapq.heappop(free)        start = max(arr[i], t)        wait[i] = start - arr[i]        heapq.heappush(free, start + srv[i])    w = wait[int(n_jobs * warm_frac):]    return dict(Wq=w.mean(), Lq=w.mean() * lam, ci=1.96 * w.std(ddof=1) / math.sqrt(len(w)))dist_rows = []for dist in ("exp", "uniform", "deterministic"):    r = simulate_general(LAM, MU, c_star, dist, seed=7)    dist_rows.append(dict(розподіл=dist, Wq_hr=r["Wq"], Lq=r["Lq"], ci95=r["ci"]))df_dist = pd.DataFrame(dist_rows)df_dist

## 11. Висновки- Аналітична (Ерланг) та імітаційна (Монте-Карло) моделі узгоджуються в межах довірчих інтервалів — модель валідна.- Функція сумарних витрат C(c) опукла, оптимум c* знайдено прямим перебором.- Зі зростанням кількості реплікацій довірчий інтервал для Wq звужується пропорційно 1/√R.- Черга зростає нелінійно (гіперболічно) при ρ → 1, тому проєктувати систему «під зав'язку» неприпустимо.- Менша варіабельність часу обслуговування (детермінований розподіл) при тому самому середньому дає меншу чергу, ніж експоненційний (ефект Полячека–Хінчина).